In [92]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report,accuracy_score,f1_score,confusion_matrix
import regex as re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
import pickle

In [93]:
data = pd.read_csv("email.csv")

In [94]:
data

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [95]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5573 non-null   object
 1   Message   5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [96]:
data.describe()

,Category,Message
count,5573,5573
unique,3,5158
top,ham,"Sorry, I'll call later"
freq,4825,30


In [97]:
data_y = data[data["Category"]!='{"mode":"full"']

In [98]:
X = data_y['Message']

In [99]:
data_y['Category'].unique()

array(['ham', 'spam'], dtype=object)

In [100]:
y = pd.get_dummies(data_y['Category'],drop_first=True).values

In [101]:
y

array([[0],
       [0],
       [1],
       ...,
       [0],
       [0],
       [0]], dtype=uint8)

In [102]:
processed_texts = []

for sentences in X:
    
    processed_text =  re.sub(r'[^a-zA-Z]',' ',str(sentences))
    
    processed_text = processed_text.lower()
    
    processed_texts.append(processed_text)
    


In [103]:
vectorizer = TfidfVectorizer(max_features=3000,stop_words=stopwords.words('english'))

X = vectorizer.fit_transform(processed_texts).toarray()

In [104]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=0)

In [105]:
model = GaussianNB()

model.fit(X_train,y_train)

c:\Users\v.rajkumar\AppData\Local\Programs\Python\Python37\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GaussianNB()

In [106]:
pred_y = model.predict(X_test)


In [107]:
print(np.concatenate((pred_y.reshape(len(pred_y),1),y_test.reshape(len(y_test),1)),axis=1))


[[0 0]
 [1 1]
 [0 0]
 ...
 [1 1]
 [1 0]
 [0 0]]


In [110]:
acc = accuracy_score(y_test,pred_y)

In [111]:
acc*100

87.50897343862168

In [112]:
c_r = classification_report(y_test,pred_y)

In [114]:
print(c_r)

              precision    recall  f1-score   support

           0       0.98      0.87      0.92      1208
           1       0.52      0.89      0.65       185

    accuracy                           0.88      1393
   macro avg       0.75      0.88      0.79      1393
weighted avg       0.92      0.88      0.89      1393



In [121]:
email = ["Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."]

vectorized_email = vectorizer.transform(email).toarray()

pred = model.predict(vectorized_email)

In [122]:
print("ham" if pred[0] == 0 else "Spam")

ham


In [124]:
with open("model.pkl",'wb') as f:
    pickle.dump(model,f)

In [126]:
with open("vectorizer.pkl",'wb') as f1:
    pickle.dump(vectorizer,f1)